# Laboratoire d'agents — assistant Jàng

Ce notebook compare deux versions du même agent qui répond aux questions sur les 14 exercices publics de Jàng :

1. **sans mémoire** : chaque question est envoyée seule ;
2. **avec mémoire de session** : tout l'historique de la conversation est renvoyé à chaque tour.

L'agent s'appuie uniquement sur `data/catalogue_public.json`. Il décrit le catalogue : il ne résout pas les exercices et ne donne aucune correction.

Kernel : l'environnement Python de `.venv` (voir `requirements-gemini.txt`).

## 0. Réglages

Le modèle, la limite de sortie et le chemin du catalogue sont des constantes : changez-les ici uniquement.

In [ ]:
MODEL = "gemini-flash-latest"  # alias du modèle « flash » le plus récent (rapide et économique)
MAX_TOKENS = 1024            # plafond large : la consigne « très concis » est dans le prompt système
CATALOGUE_PATH = "data/catalogue_public.json"  # relatif à la racine du projet

## 1. Chargement de la clé API

La clé est lue depuis le fichier `.env` (ignoré par git) grâce à `python-dotenv`. Elle n'est jamais écrite dans le notebook ni affichée.
Si elle est absente, une erreur explicite en français est levée.

In [ ]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(
        "GEMINI_API_KEY est introuvable. Ajoutez la ligne GEMINI_API_KEY=... dans le fichier .env "
        "à la racine du projet, puis relancez le kernel et ce notebook."
    )

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Si l'alias n'est pas disponible pour ce compte, on prend le modèle « flash » le plus récent proposé.
try:
    client.models.get(model=MODEL)
except errors.ClientError:
    flash = sorted(
        m.name.removeprefix("models/")
        for m in client.models.list()
        if "flash" in m.name and "generateContent" in (m.supported_actions or [])
        and not any(x in m.name for x in ("lite", "image", "tts", "audio", "live", "preview", "exp"))
    )
    if not flash:
        raise RuntimeError("Aucun modèle « flash » disponible pour cette clé.")
    MODEL = flash[-1]
print("Clé API chargée depuis l'environnement. Modèle :", MODEL)

## 2. Catalogue et prompt système

On charge le catalogue (ID, chapitre, énoncé, statut de validation — sans réponses) et on l'insère dans le prompt système avec des règles strictes :
décrire sans inventer, ne jamais résoudre ni corriger, demander une précision si la question est ambiguë.

In [ ]:
import json

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / CATALOGUE_PATH).exists())
catalogue = json.loads((root / CATALOGUE_PATH).read_text(encoding="utf-8"))
print(len(catalogue), "exercices chargés depuis", root / CATALOGUE_PATH)

SYSTEM_PROMPT = """Tu es l'assistant Jàng. Tu aides à parcourir le catalogue public des exercices de Jàng.

Règles :
- Tu t'appuies uniquement sur le catalogue ci-dessous (ID, chapitre, énoncé, statut de validation). Tu n'utilises aucune autre source.
- Tu décris le catalogue : tu ne résous pas les exercices et tu ne donnes ni réponse, ni indice de résolution, ni correction. Tu n'inventes rien.
- Si l'information n'est pas dans le catalogue, ou si la question est ambiguë (par exemple « le deuxième » sans liste évoquée plus tôt dans la conversation), dis-le simplement et demande une précision.
- Quand tu énumères des exercices ou des chapitres, suis l'ordre du catalogue, cite l'ID, et écris le titre du chapitre sans le préfixe de matière.
- Réponds en français, de façon très concise (moins de 120 tokens), sans introduction.

<catalogue>
""" + json.dumps(catalogue, ensure_ascii=False, indent=1) + """
</catalogue>"""

## 3. Fonction d'appel commune

Les deux agents partagent le même appel à l'API du modèle. La seule différence est la liste `messages` qu'on lui envoie.
Les erreurs courantes (clé invalide, quota, réseau) sont traduites en messages français, et on signale une réponse tronquée par la limite de tokens.

In [ ]:
import time


def call_model(messages):
    """Envoie `messages` (rôles user / assistant) au modèle et renvoie le texte de la réponse."""
    contents = [
        types.Content(role="user" if m["role"] == "user" else "model", parts=[types.Part(text=m["content"])])
        for m in messages
    ]
    for attempt in range(5):  # le serveur gratuit peut être saturé (503) : on réessaie avec une attente croissante
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    max_output_tokens=MAX_TOKENS,
                ),
            )
            break
        except (errors.ServerError, errors.ClientError) as e:
            retryable = e.code in (429, 500, 503)
            if retryable and attempt < 4:
                time.sleep(5 * 2 ** attempt)
                continue
            if e.code == 429:
                raise RuntimeError("Limite de requêtes atteinte : patientez une minute puis réessayez.") from e
            if e.code in (400, 401, 403):
                raise RuntimeError(f"Clé API refusée ou requête invalide (code {e.code}) : {e.message}") from e
            raise RuntimeError(f"Erreur de l'API (code {e.code}) : {e.message}") from e

    text = response.text or ""
    note = f"[{response.usage_metadata.candidates_token_count} tokens de sortie]"
    if response.candidates and str(response.candidates[0].finish_reason).endswith("MAX_TOKENS"):
        note += f" ⚠️ réponse tronquée par MAX_TOKENS={MAX_TOKENS}"
    return text, note

## 4. Agent SANS mémoire

À chaque tour, on n'envoie que la question courante : `messages = [question]`.
L'API est sans état, donc l'agent ne sait rien de l'échange précédent.

In [ ]:
def ask_without_memory(question):
    text, note = call_model([{"role": "user", "content": question}])
    print("Q :", question)
    print("R :", text)
    print(note, "\n")

In [ ]:
ask_without_memory("Quels chapitres de chimie couvre Jàng ?")
ask_without_memory("Et le deuxième, c'est quel exercice ?")

**À observer :** la première réponse liste les chapitres de chimie. À la seconde, l'agent n'a plus cette liste sous les yeux :
il ne peut pas savoir à quoi « le deuxième » se rapporte, et doit demander une précision plutôt que d'inventer.

## 5. Agent AVEC mémoire de session

La mémoire est simplement une liste `history` qui contient tous les messages échangés.
À chaque tour, on y ajoute la question, on envoie **tout** l'historique, puis on y ajoute la réponse de l'agent.

In [ ]:
history = []


def ask_with_memory(question):
    history.append({"role": "user", "content": question})
    try:
        text, note = call_model(history)
    except Exception:
        history.pop()  # garde l'historique cohérent si l'appel échoue
        raise
    history.append({"role": "assistant", "content": text})
    print("Q :", question)
    print("R :", text)
    print(note, "\n")

In [ ]:
ask_with_memory("Quels chapitres de chimie couvre Jàng ?")
ask_with_memory("Et le deuxième, c'est quel exercice ?")

**À observer :** cette fois, l'agent retrouve « le deuxième » dans sa propre liste et répond **JNG-PC-02** (Acides forts).

In [ ]:
print(len(history), "messages dans l'historique de session :")
for message in history:
    print("-", message["role"], ":", message["content"][:70].replace("\n", " "))

## Pour aller plus loin

- La mémoire n'existe que dans la variable `history` : elle disparaît si on redémarre le kernel.
- L'historique est renvoyé en entier à chaque tour, donc le coût en tokens d'entrée augmente avec la durée de la conversation.
- Dans les deux cas, le prompt système impose à l'agent de ne décrire que le catalogue : pas de résolution, pas de correction.